In [1]:
import pandera.polars as pa
import numpy as np
from pathlib import Path
import polars as pl

In [2]:
def LengthCheck(size: int) -> pa.Check:
    return pa.Check(lambda x: x.apply(len) == size, error=f'List must have length {size}')


data_datasframe_schema = pa.DataFrameSchema({
    'id': pa.Column(
        pl.UInt64,
        unique=True,
        checks=[
            pa.Check.ge(0),
        ]),
    'target': pa.Column(
        pl.UInt8,
        checks=[
            pa.Check.ge(0),
            pa.Check.le(1),
        ]),
    'OfflineElectronTriggerContainer.lhmedium': pa.Column(
        pl.Boolean,
    ),
    'OfflineElectronTriggerContainer.lhvloose': pa.Column(
        pl.Boolean,
    ),
    'TrigEMClusterContainer.eta': pa.Column(
        pl.Float32,
        checks=[
            pa.Check.ge(-2.5),
            pa.Check.le(2.5),
        ]),
    'TrigEMClusterContainer.et': pa.Column(
        pl.Float32,
        checks=[
            pa.Check.ge(0),
            pa.Check.le(1e12),
        ]),
    'TrigEMClusterContainer.ringsE': pa.Column(
        pl.List(pl.Float32),
    ),
})

In [3]:
def generate_example(n_rows: int, id_start: int = 0) -> pl.DataFrame:
    data = {
    'id': np.arange(id_start, id_start + n_rows, dtype=np.uint64),
    'target': np.random.randint(0, 2, n_rows, dtype=np.uint8),
    'OfflineElectronTriggerContainer.lhmedium': np.random.choice([True, False], n_rows),
    'OfflineElectronTriggerContainer.lhvloose': np.random.choice([True, False], n_rows),
    'TrigEMClusterContainer.eta': np.random.uniform(-2.5, 2.5, n_rows).astype(np.float32),
    'TrigEMClusterContainer.et': np.random.uniform(12e3, 100e3, n_rows).astype(np.float32),
    'TrigEMClusterContainer.ringsE': [np.random.uniform(0, 1e6, 100).tolist() for _ in range(n_rows)]
    }
    example_df = pl.DataFrame(data).with_columns(pl.col('TrigEMClusterContainer.ringsE').cast(pl.List(pl.Float32)))
    return example_df

In [4]:
output_dir = Path('..') / 'tests' / 'data' / 'test_dataset' / 'electron_ringer.parquet'
output_dir.mkdir(parents=True, exist_ok=True)

In [5]:
# Generate example data with PyArrow dtypes
n_rows = 100000
for i in range(2):
    example_df = generate_example(n_rows, id_start=i*n_rows)
    example_df.write_parquet(output_dir / f'{i:04d}.parquet')
example_df

id,target,OfflineElectronTriggerContainer.lhmedium,OfflineElectronTriggerContainer.lhvloose,TrigEMClusterContainer.eta,TrigEMClusterContainer.et,TrigEMClusterContainer.ringsE
u64,u8,bool,bool,f32,f32,list[f32]
100000,0,false,false,-1.748538,29721.082031,"[289146.6875, 445297.34375, … 571884.25]"
100001,0,false,true,2.260221,94592.5625,"[406272.40625, 610574.625, … 733335.5625]"
100002,1,false,true,1.154022,69357.507812,"[706931.75, 28901.347656, … 768813.5625]"
100003,0,true,false,-1.390526,63887.613281,"[143912.390625, 134645.515625, … 617394.375]"
100004,0,false,false,-2.432841,61345.820312,"[703203.0, 457399.03125, … 346588.71875]"
…,…,…,…,…,…,…
199995,0,false,false,0.169625,95067.390625,"[794640.6875, 9008.082031, … 786292.5]"
199996,1,false,false,2.252737,45182.511719,"[395532.40625, 247247.15625, … 647833.8125]"
199997,0,true,true,2.287133,40848.5,"[386631.84375, 465060.9375, … 660438.25]"


In [6]:
df = pl.read_parquet(
    '../tests/data/test_dataset/electron_ringer.parquet'
)

In [7]:
data_datasframe_schema.validate(df)

id,target,OfflineElectronTriggerContainer.lhmedium,OfflineElectronTriggerContainer.lhvloose,TrigEMClusterContainer.eta,TrigEMClusterContainer.et,TrigEMClusterContainer.ringsE
u64,u8,bool,bool,f32,f32,list[f32]
0,0,false,false,-0.040649,86566.6875,"[138812.4375, 245184.9375, … 848177.5]"
1,0,true,false,1.343226,56607.734375,"[916511.4375, 198649.515625, … 598297.75]"
2,1,false,false,2.366676,14107.273438,"[701822.5625, 760042.125, … 117977.867188]"
3,0,false,false,-1.054934,91434.773438,"[415156.3125, 523452.125, … 982994.0]"
4,0,false,true,-0.985626,90879.75,"[990425.875, 428619.5625, … 17480.621094]"
…,…,…,…,…,…,…
199995,0,false,false,0.169625,95067.390625,"[794640.6875, 9008.082031, … 786292.5]"
199996,1,false,false,2.252737,45182.511719,"[395532.40625, 247247.15625, … 647833.8125]"
199997,0,true,true,2.287133,40848.5,"[386631.84375, 465060.9375, … 660438.25]"
